# DTaaS Localhost Flex-Cell Lab (Clean)

This version uses a container-first execution flow.

Assumptions:
- Docker is running
- Git is available
- Internet access is available

## 1) Set Variables Once

In [ ]:
LAB_HOME = "."
ZIP_FILE = "dtaas-workspace-on-localhost-v1.0-beta.zip"
UNZIP_DIR_NAME = "dtaas-workspace-on-localhost-v1.0-beta"
DTAAS_REPO_URL = "https://github.com/into-cps-association/DTaaS.git"
DTAAS_REPO_DIR = "DTaaS"
DTAAS_EXAMPLES_REPO_URL = "https://github.com/into-cps-association/DTaaS-examples.git"

print('LAB_HOME =', LAB_HOME)
print('ZIP_FILE =', ZIP_FILE)
print('UNZIP_DIR_NAME =', UNZIP_DIR_NAME)

## 2) Unzip Localhost Package

In [ ]:
%%bash -e -s "$LAB_HOME" "$ZIP_FILE" "$UNZIP_DIR_NAME"
LAB_HOME="$1"
ZIP_FILE="$2"
UNZIP_DIR_NAME="$3"

if [ ! -f "$ZIP_FILE" ]; then
  echo "Zip file not found: $ZIP_FILE"
  exit 1
fi

if [ -d "$LAB_HOME/$UNZIP_DIR_NAME" ]; then
  echo "Already extracted: $LAB_HOME/$UNZIP_DIR_NAME"
else
  unzip -o "$ZIP_FILE" -d "$LAB_HOME"
fi

ls -la "$LAB_HOME" | head -n 30

## 3) Start DTaaS Localhost

In [ ]:
%%bash -e -s "$LAB_HOME"
LAB_HOME="$1"

cd "$LAB_HOME"
cp -f .env.example .env
cp -f config/dex-config.yaml.example config/dex-config.yaml
docker compose up -d
docker compose ps

## 4) Open DTaaS UI
Open http://localhost and log in with:
- user@intocps.org
- user

## 5) Clone Original DTaaS Repository (Optional)

In [ ]:
%%bash -e -s "$LAB_HOME" "$DTAAS_REPO_URL" "$DTAAS_REPO_DIR"
LAB_HOME="$1"
DTAAS_REPO_URL="$2"
DTAAS_REPO_DIR="$3"

cd "$LAB_HOME"
if [ -d "$DTAAS_REPO_DIR/.git" ]; then
  echo "Already cloned: $DTAAS_REPO_DIR"
else
  git clone "$DTAAS_REPO_URL" "$DTAAS_REPO_DIR"
fi
cd "$DTAAS_REPO_DIR"
git rev-parse --short HEAD

## 6) Clone DTaaS-examples into Mounted Workspace
This avoids any manual sync. Container sees this as /workspace/examples.

In [ ]:
%%bash -e -s "$LAB_HOME" "$DTAAS_EXAMPLES_REPO_URL"
LAB_HOME="$1"
DTAAS_EXAMPLES_REPO_URL="$2"

EXAMPLES_HOST_DIR="$LAB_HOME/files/user/examples"
if [ -d "$EXAMPLES_HOST_DIR/.git" ]; then
  echo "Already cloned: $EXAMPLES_HOST_DIR"
else
  git clone "$DTAAS_EXAMPLES_REPO_URL" "$EXAMPLES_HOST_DIR"
fi

find "$EXAMPLES_HOST_DIR" -maxdepth 3 -type d -name flex-cell

## 7) Bootstrap Flex-Cell Dependencies in Container
Run once. Uses uv for Python dependencies.

In [ ]:
%%bash -e
USER_CONTAINER="$(docker ps --filter label=com.docker.compose.service=user --format '{{.Names}}' | head -n 1)"
if [ -z "$USER_CONTAINER" ]; then
  echo "User workspace container not running. Start DTaaS localhost first."
  exit 1
fi

docker exec "$USER_CONTAINER" bash -lc '
set -e
retry() {
  n=0
  until "$@"; do
    n=$((n+1))
    if [ "$n" -ge 6 ]; then
      echo "Command failed after retries: $*"
      return 1
    fi
    sleep 5
  done
}

retry apt-get update
retry apt-get install -y default-jre maven libguice-java unzip zip build-essential g++ gcc python3-dev pkg-config curl ca-certificates
mvn -f /workspace/examples/tools/TwinManager/pom.xml package -Dmaven.test.skip=true -Dmaven.javadoc.skip=true
cp /workspace/examples/tools/TwinManager/target/TwinManagerFramework-0.0.2.jar /workspace/examples/tools/
if ! command -v uv >/dev/null 2>&1; then
  curl -LsSf https://astral.sh/uv/install.sh | sh
fi
export PATH="$HOME/.local/bin:$PATH"
uv python install 3.11
uv venv --clear --python 3.11 /workspace/examples/tools/flex-cell/requirements/venv
source /workspace/examples/tools/flex-cell/requirements/venv/bin/activate
uv pip install -r /workspace/examples/tools/flex-cell/requirements/requirements.txt
uv pip install pyhocon
uv pip install -e /workspace/examples/tools/flex-cell/robots_flexcell
uv pip install -e /workspace/examples/tools/flex-cell/kukalbrinterface
mkdir -p /workspace/examples/data/flex-cell/output /workspace/examples/data/flex-cell/input /workspace/examples/data/flex-cell/input/physical_twin
'

## 8) Run prepare
Edit this host file first and set broker credentials:
files/user/examples/data/flex-cell/input/connections.conf

In [ ]:
%%bash -e
USER_CONTAINER="$(docker ps --filter label=com.docker.compose.service=user --format '{{.Names}}' | head -n 1)"
docker exec "$USER_CONTAINER" bash -lc 'cd /workspace/examples/digital_twins/flex-cell && ./lifecycle/prepare'

## 9) Run execute
Let it run for 20-30 seconds, then interrupt the cell.

In [ ]:
%%bash
USER_CONTAINER="$(docker ps --filter label=com.docker.compose.service=user --format '{{.Names}}' | head -n 1)"
docker exec "$USER_CONTAINER" bash -lc 'cd /workspace/examples/digital_twins/flex-cell && ./lifecycle/execute'

## 10) Save Results and Analyze

In [ ]:
import subprocess
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
subprocess.run(
    """USER_CONTAINER="$(docker ps --filter label=com.docker.compose.service=user --format '{{.Names}}' | head -n 1)";
    docker exec "$USER_CONTAINER" bash -lc 'set -e; cd /workspace/examples/digital_twins/flex-cell; ./lifecycle/save; ./lifecycle/analyze'""",
    shell=True,
    check=True
)

# This is exactly where analyze writes inside the container
OUT_DIR_CONTAINER = Path("/workspace/examples/data/flex-cell/output")

# Host-side mapping of /workspace/examples -> files/user/examples (relative to notebook folder)
workspace_examples_host = Path("files/user/examples")
out_dir_host = workspace_examples_host / OUT_DIR_CONTAINER.relative_to("/workspace/examples")

plot_file = out_dir_host / "experiment_plot.png"

if not plot_file.exists():
    raise FileNotFoundError(f"Plot not found at: {plot_file}")

img = mpimg.imread(plot_file)
plt.figure(figsize=(14, 8))
plt.imshow(img)
plt.axis("off")
plt.show()

## 12) Terminate and Clean

In [ ]:
%%bash
set -euo pipefail

USER_CONTAINER="$(docker ps --filter label=com.docker.compose.service=user --format '{{.Names}}' | head -n 1)"
[ -n "$USER_CONTAINER" ] || { echo "No user container running"; exit 1; }

docker exec "$USER_CONTAINER" bash -lc '
cd /workspace/examples/digital_twins/flex-cell
./lifecycle/terminate || echo "terminate skipped (already stopped)"
./lifecycle/clean || echo "clean skipped (already clean)"
'

## 13) Shut Down DTaaS Localhost

In [ ]:
%%bash -e -s "$LAB_HOME"
LAB_HOME="$1"
cd "$LAB_HOME"
docker compose down

## Comprehension Questions
1. Why is prepare required before execute in this flex-cell workflow?
2. What does save preserve that is useful across multiple runs?